# 📊 Direct Database 20-Feature Extractor from `Order_Details`

This notebook directly connects to the MySQL Database schema (`Customer_Details` and `Order_Details` tables) without relying on any external helper files.

It accepts any `customer_id`, verifies if the customer exists in the database, and calculates **exactly the 20 SHAP features** required for prediction.

### 🎯 The 20 Required Features:
1. `min_pp`: Minimum Product Price
2. `p90_pp`: 90th Percentile Product Price
3. `p95_pad`: 95th Percentile Price After Discount
4. `p25_pp`: 25th Percentile Product Price
5. `p95_pp`: 95th Percentile Product Price
6. `max_pp`: Maximum Product Price
7. `min_pad`: Minimum Price After Discount
8. `p75_pp`: 75th Percentile Product Price
9. `max_pad`: Maximum Price After Discount
10. `p90_pad`: 90th Percentile Price After Discount
11. `avg_pp`: Average Product Price
12. `p50_pp`: 50th Percentile (Median) Product Price
13. `p25_pad`: 25th Percentile Price After Discount
14. `p50_pad`: 50th Percentile (Median) Price After Discount
15. `avg_pad`: Average Price After Discount
16. `p75_pad`: 75th Percentile Price After Discount
17. `p90_pd`: 90th Percentile Product Discount
18. `total_pad`: Total Sum of Price After Discount
19. `max_pd`: Maximum Product Discount
20. `p95_pd`: 95th Percentile Product Discount

In [4]:
import os
import pickle
import numpy as np
import pandas as pd
import mysql.connector
from mysql.connector import Error

# Direct MySQL Connection Parameters
MYSQL_HOST = os.environ.get("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", 3306))
MYSQL_USER = os.environ.get("MYSQL_USER", "root")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "root123")
MYSQL_DATABASE = os.environ.get("MYSQL_DATABASE", "farmora")

# Exact 20 SHAP features in required prediction order
SELECTED_20_FEATURES = [
    'min_pp', 'p90_pp', 'p95_pad', 'p25_pp', 'p95_pp', 'max_pp',
    'min_pad', 'p75_pp', 'max_pad', 'p90_pad', 'avg_pp', 'p50_pp',
    'p25_pad', 'p50_pad', 'avg_pad', 'p75_pad', 'p90_pd', 'total_pad',
    'max_pd', 'p95_pd'
]

def get_direct_db_connection():
    """Establish direct connection to MySQL database without external files."""
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            port=MYSQL_PORT,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=MYSQL_DATABASE,
            autocommit=True
        )
        return conn
    except Error:
        try:
            conn = mysql.connector.connect(
                host=MYSQL_HOST,
                port=MYSQL_PORT,
                user=MYSQL_USER,
                password=MYSQL_PASSWORD,
                database="organic_food_traceability",
                autocommit=True
            )
            return conn
        except Error as e:
            print(f"❌ Could not connect to MySQL database: {e}")
            return None

print("✅ Setup completed. Features list:")
print(SELECTED_20_FEATURES)

✅ Setup completed. Features list:
['min_pp', 'p90_pp', 'p95_pad', 'p25_pp', 'p95_pp', 'max_pp', 'min_pad', 'p75_pp', 'max_pad', 'p90_pad', 'avg_pp', 'p50_pp', 'p25_pad', 'p50_pad', 'avg_pad', 'p75_pad', 'p90_pd', 'total_pad', 'max_pd', 'p95_pd']


In [7]:
def get_customer_prediction_features(customer_id):
    """
    Directly queries MySQL database schema (Customer_Details & Order_Details).
    If customer is not in the database, prints:
    'Customer is not in the current database'
    """
    conn = get_direct_db_connection()
    if not conn:
        print("❌ Unable to connect to MySQL database.")
        return None
    
    cursor = conn.cursor(dictionary=True)
    
    try:
        # 1. Direct query to check Customer_Details table
        cursor.execute(
            "SELECT customer_id, customer_name, email_id FROM Customer_Details WHERE customer_id = %s;",
            (int(customer_id),)
        )
        cust_row = cursor.fetchone()
        
        if not cust_row:
            print("Customer is not in the current database")
            cursor.close()
            conn.close()
            return None
        
        # 2. Direct query to fetch Order_Details for customer
        query_orders = """
            SELECT 
                order_id, customer_id, product_id, product_count,
                product_price, product_discount, price_after_discount,
                order_date
            FROM Order_Details
            WHERE customer_id = %s;
        """
        cursor.execute(query_orders, (int(customer_id),))
        orders = cursor.fetchall()
        
        cursor.close()
        conn.close()
        
        if not orders:
            print("Customer is not in the current database")
            return None
        
        # 3. Load transactions into pandas DataFrame
        df_orders = pd.DataFrame(orders)
        df_orders['product_price'] = df_orders['product_price'].astype(float)
        df_orders['product_discount'] = df_orders['product_discount'].astype(float)
        df_orders['price_after_discount'] = df_orders['price_after_discount'].astype(float)
        
        pp = df_orders['product_price'].values
        pd_disc = df_orders['product_discount'].values
        pad = df_orders['price_after_discount'].values
        
        # 4. Compute exact 20 SHAP features
        feature_dict = {
            'min_pp': float(np.min(pp)),
            'p90_pp': float(np.percentile(pp, 90)),
            'p95_pad': float(np.percentile(pad, 95)),
            'p25_pp': float(np.percentile(pp, 25)),
            'p95_pp': float(np.percentile(pp, 95)),
            'max_pp': float(np.max(pp)),
            'min_pad': float(np.min(pad)),
            'p75_pp': float(np.percentile(pp, 75)),
            'max_pad': float(np.max(pad)),
            'p90_pad': float(np.percentile(pad, 90)),
            'avg_pp': float(np.mean(pp)),
            'p50_pp': float(np.percentile(pp, 50)),
            'p25_pad': float(np.percentile(pad, 25)),
            'p50_pad': float(np.percentile(pad, 50)),
            'avg_pad': float(np.mean(pad)),
            'p75_pad': float(np.percentile(pad, 75)),
            'p90_pd': float(np.percentile(pd_disc, 90)),
            'total_pad': float(np.sum(pad)),
            'max_pd': float(np.max(pd_disc)),
            'p95_pd': float(np.percentile(pd_disc, 95))
        }
        
        features_df = pd.DataFrame([feature_dict])[SELECTED_20_FEATURES]
        print(f"✅ Extracted 20 features for Customer #{customer_id} ({cust_row['customer_name']}):")
        return features_df
        
    except Error as e:
        print(f"❌ Database Query Error: {e}")
        if cursor:
            cursor.close()
        if conn:
            conn.close()
        return None

In [8]:
# Set any Customer ID to extract 20 prediction features
customer_id_input = 1

result_df = get_customer_prediction_features(customer_id_input)
if result_df is not None:
    display(result_df)

✅ Extracted 20 features for Customer #1 (Stefanie Y Frye):


,min_pp,p90_pp,p95_pad,p25_pp,p95_pp,max_pp,min_pad,p75_pp,max_pad,p90_pad,avg_pp,p50_pp,p25_pad,p50_pad,avg_pad,p75_pad,p90_pd,total_pad,max_pd,p95_pd
0,12.21,91.026,79.432,32.72,93.514,95.41,9.65,69.03,90.64,71.178,52.512308,43.6,24.86,35.31,42.835385,55.17,24.8,556.86,30.0,27.0


In [10]:
# Test with a non-existent Customer ID
missing_customer_id = 9999000000000000000000000000000000000000000000000000000000000

print(f"Testing Customer ID: {missing_customer_id}")
get_customer_prediction_features(missing_customer_id)

Testing Customer ID: 9999000000000000000000000000000000000000000000000000000000000
Customer is not in the current database


In [11]:
# Optional: Run LightGBM model prediction test with the extracted features
if result_df is not None:
    model_file = 'lgbm_multiclass_model.pkl'
    encoder_file = 'label_encoder.pkl'
    
    if os.path.exists(model_file) and os.path.exists(encoder_file):
        with open(model_file, 'rb') as f:
            lgb_model = pickle.load(f)
        with open(encoder_file, 'rb') as f:
            label_enc = pickle.load(f)
            
        probabilities = lgb_model.predict_proba(result_df)[0]
        top1_idx = np.argmax(probabilities)
        predicted_prod_id = label_enc.inverse_transform([top1_idx])[0]
        conf_score = probabilities[top1_idx] * 100
        
        print(f"\n🔮 LightGBM Prediction for Customer #{customer_id_input}:")
        print(f"--> Recommended Product ID: #{predicted_prod_id}")
        print(f"--> Confidence Score: {conf_score:.2f}%")


🔮 LightGBM Prediction for Customer #1:
--> Recommended Product ID: #57
--> Confidence Score: 76.60%
